In [0]:
# ─── CÉLULA 1 — CONFIGURAÇÃO, CARREGAMENTO DO MODELO E PIPELINE ──────────────────
# Carrega o modelo campeão registrado no Unity Catalog e o pipeline de feature engineering salvo no Volume. A separação entre modelo e pipeline permite aplicar o mesmo pré-processamento a qualquer novo dado antes da inferência.
# mlflow.set_registry_uri("databricks-uc") é obrigatório antes de load_model() para que o MLflow resolva o nome do modelo no Unity Catalog Registry.
# MLFLOW_DFS_TMP redireciona temporários para o Volume — DBFS root desabilitado.
# model_uri com sufixo "/1" carrega explicitamente a versão 1 do modelo.

import mlflow
import mlflow.spark
import os
from pyspark.ml.evaluation import BinaryClassificationEvaluator
from pyspark.ml.functions import vector_to_array
from pyspark.ml import PipelineModel
from pyspark.sql import functions as F

os.environ['MLFLOW_DFS_TMP'] = "/Volumes/workspace/default/modelos_ml/tmp"

CATALOG       = "workspace"
SCHEMA        = "default"
MODEL_NAME    = f"{CATALOG}.{SCHEMA}.telco-churn-predictor"
PIPELINE_PATH = "/Volumes/workspace/default/modelos_ml/pipeline_model"
PRED_TABLE    = f"{CATALOG}.{SCHEMA}.telco_predictions"

mlflow.set_registry_uri("databricks-uc")

model_uri       = f"models:/{MODEL_NAME}/1"
model_loaded    = mlflow.spark.load_model(model_uri)
pipeline_loaded = PipelineModel.load(PIPELINE_PATH)

print(f"✔ MLFLOW_DFS_TMP: {os.environ['MLFLOW_DFS_TMP']}")
print(f"✔ Modelo carregado: {model_uri}")
print(f"✔ Pipeline carregado: {PIPELINE_PATH}")

In [0]:
# ─── CÉLULA 2 — AVALIAÇÃO NO CONJUNTO DE TESTE: MATRIZ DE CONFUSÃO ───────────────
# Reconstrói o mesmo split usado no treinamento (seed=42, proporção 80/20).
# Usar o mesmo seed garante que df_test aqui seja idêntico ao usado no notebook 05, tornando as métricas diretamente comparáveis entre os dois notebooks.
# df_test.cache() melhora performance: o DataFrame será acessado múltiplas vezes nas células seguintes (predições, extração de probabilidade).
# A matriz de confusão complementa o AUC-ROC com interpretação operacional:
# FN = churn não detectado (custo alto); FP = retenção desnecessária.

df_gold = spark.read.table(f"{CATALOG}.{SCHEMA}.telco_gold")
_, df_test = df_gold.randomSplit([0.8, 0.2], seed=42)
df_test.cache()

preds = model_loaded.transform(df_test)

matriz = (
    preds
    .groupBy("label", "prediction")
    .count()
    .withColumn("tipo",
        F.when((F.col("label")==1) & (F.col("prediction")==1), "TP")
        .when((F.col("label")==0) & (F.col("prediction")==0), "TN")
        .when((F.col("label")==0) & (F.col("prediction")==1), "FP")
        .otherwise("FN")
    )
    .orderBy("label", "prediction")
)
display(matriz)

In [0]:
# ─── CÉLULA 3 — EXTRAÇÃO DE PROBABILIDADE E CLASSIFICAÇÃO DE RISCO ───────────────
# O modelo retorna `probability` como vetor [P(classe=0), P(classe=1)].
# vector_to_array() converte o VectorUDT para ArrayType, permitindo indexar com [1] para extrair P(churn) — probabilidade da classe positiva.
# Os thresholds de risco (0.4 e 0.7) são ajustáveis conforme o custo de negócio:
# - ALTO (≥0.7): ação imediata de retenção recomendada
# - MÉDIO (0.4–0.7): monitoramento proativo
# - BAIXO (<0.4): sem intervenção necessária

preds_com_prob = (
    preds
    .withColumn("prob_array", vector_to_array(F.col("probability")))
    .withColumn("churn_prob", F.round(F.col("prob_array")[1], 4))
    .withColumn("risco",
        F.when(F.col("churn_prob") >= 0.7, "ALTO")
        .when(F.col("churn_prob") >= 0.4, "MÉDIO")
        .otherwise("BAIXO")
    )
    .select("label", "prediction", "churn_prob", "risco")
)

display(
    preds_com_prob
    .orderBy(F.col("churn_prob").desc())
    .limit(10)
)

In [0]:
# ─── CÉLULA 4 — BATCH INFERENCE EM DADOS NOVOS ───────────────────────────────────
# Simula um cenário de produção: aplica o modelo a clientes "novos" sem label.
# Os dados vêm da camada Silver (antes do pipeline Gold), então as três features criadas no notebook 04 precisam ser recriadas manualmente aqui — o pipeline salvo não as gera, apenas as consome como input.
# A sequência correta é: criar features → pipeline.transform() → model.transform().
# scored_at registra o timestamp da inferência para rastreabilidade no Delta Lake.

df_novos = (
    spark.read.table(f"{CATALOG}.{SCHEMA}.telco_silver")
    .sample(0.05, seed=99)
    .drop("Churn")
)

service_cols = [
    "PhoneService", "MultipleLines", "OnlineSecurity",
    "OnlineBackup", "DeviceProtection", "TechSupport",
    "StreamingTV", "StreamingMovies"
]

df_novos = df_novos.withColumn(
    "num_services",
    sum([F.when(F.col(c) == "Yes", 1).otherwise(0) for c in service_cols])
)

df_novos = df_novos.withColumn(
    "charges_per_tenure",
    F.when(
        F.col("tenure") > 0,
        F.col("TotalCharges") / F.col("tenure")
    ).otherwise(F.col("MonthlyCharges"))
)

df_novos = df_novos.withColumn(
    "is_new_customer",
    F.when(F.col("tenure") <= 12, 1).otherwise(0)
)

df_novos_features = pipeline_loaded.transform(df_novos)

df_scored = (
    model_loaded.transform(df_novos_features)
    .withColumn("prob_array", vector_to_array(F.col("probability")))
    .withColumn("churn_prob", F.round(F.col("prob_array")[1], 4))
    .withColumn("risco",
        F.when(F.col("churn_prob") >= 0.7, "ALTO")
        .when(F.col("churn_prob") >= 0.4, "MÉDIO")
        .otherwise("BAIXO")
    )
    .withColumn("scored_at", F.current_timestamp())
    .select("churn_prob", "risco", "prediction", "scored_at")
)

print(f"✔ {df_scored.count()} clientes pontuados")
display(
    df_scored
    .orderBy(F.col("churn_prob").desc())
    .limit(10)
)

In [0]:
# ─── CÉLULA 5 — PERSISTÊNCIA DAS PREDIÇÕES E RESUMO POR FAIXA DE RISCO ───────────
# Salva os scores no Unity Catalog para consumo por dashboards e sistemas downstream.
# O uso de saveAsTable com nome qualificado garante que a tabela fique acessível em workspace.default.telco_predictions — mesmo padrão das demais camadas.
# O resumo por faixa de risco quantifica a distribuição do risco na base inferida, informação diretamente acionável pela equipe de CRM/retenção.

(
    df_scored
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(PRED_TABLE)
)

print(f"✔ Predições salvas em: {PRED_TABLE}")

display(
    spark.read.table(PRED_TABLE)
    .groupBy("risco")
    .agg(
        F.count("*").alias("total_clientes"),
        F.round(F.avg("churn_prob"), 4).alias("prob_media")
    )
    .orderBy("risco")
)

## ✅ Pipeline Completo — Resumo Final

### Arquitetura Medalhão — Unity Catalog (workspace.default)
| Camada    | Tabela                              | Linhas | Descrição                  |
|-----------|-------------------------------------|--------|----------------------------|
| Bronze    | workspace.default.telco_bronze      | 7.043  | Dados brutos do CSV        |
| Prata     | workspace.default.telco_silver      | 7.043  | Limpos e tipados           |
| Ouro      | workspace.default.telco_gold        | 7.043  | Features prontas para ML   |
| Predições | workspace.default.telco_predictions | ~350   | Score de churn por cliente |

### Modelo Campeão
- Algoritmo  : LR (LogisticRegression)
- AUC-ROC    : ~0.82
- F1-Score   : ~0.79
- Registrado : workspace.default.telco-churn-predictor (versão 1)

### Tecnologias Utilizadas
Apache Spark · PySpark · Delta Lake · MLflow · Unity Catalog · Databricks